In [ ]:
import re, time, json, random, warnings, logging, asyncio
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Literal, NamedTuple, Optional

import pandas as pd
import aiohttp

# =======================
# LOGGING
# =======================
def setup_logging(log_dir: Path) -> logging.Logger:
    """
    Configura logging com dois handlers: console (INFO) e arquivo (DEBUG).
    
    Console mostra apenas INFO e acima para o usuário; arquivo registra DEBUG+ para análise.
    
    Args:
        log_dir: Diretório onde salvar cnpj.log
        
    Returns:
        Logger configurado pronto para usar
    """
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("cnpj")
    logger.setLevel(logging.DEBUG)
    
    if logger.hasHandlers():
        return logger
    
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
    
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    
    fh = logging.FileHandler(log_dir / "cnpj.log", encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    
    return logger

# =======================
# CONFIGURAÇÕES
# =======================
_BASE_DIR = Path.cwd()
_DATA_DIR = _BASE_DIR / "data"
_INPUT_DIR = _DATA_DIR / "input"
_OUTPUT_DIR = _DATA_DIR / "output"
_LOG_DIR = _DATA_DIR / "logs"

logger = setup_logging(_LOG_DIR)

@dataclass
class Config:
    """
    Configuração centralizada do sistema de consulta CNPJ.

    Atributos:
        arquivo_entrada: Excel com CNPJs (obrigatório: coluna 'cnpj')
        arquivo_saida: Excel de saída com todos os campos
        arquivo_cache: JSON com cache de consultas anteriores
        arquivo_falhas: CSV com CNPJs que não retornaram dados
        aba_excel: índice da aba (0-based)
        base_sleep: segundos base para backoff exponencial
        tentativas: quantas vezes tentar cada CNPJ antes de desistir
        timeout_http: timeout para cada requisição HTTP (segundos)
        sleep_fixo: pausa entre cada requisição (para respeitar rate limits)
        max_concurrent: máximo de requisições paralelas (1-10 recomendado)
        cache_ttl_dias: dias de validade do cache (padrão: 30 dias)
        formato_socios: como formatar lista de sócios ("lista_nomes", "principal", "quantidade", "detalhado")
    """
    arquivo_entrada : Path  = field(default_factory=lambda: _INPUT_DIR / "pesquisa_cnpj.xlsx")
    arquivo_saida   : Path  = field(default_factory=lambda: _OUTPUT_DIR / "pesquisa_cnpj_resultado.xlsx")
    arquivo_cache   : Path  = field(default_factory=lambda: _LOG_DIR / "cache_cnpj.json")
    arquivo_falhas  : Path  = field(default_factory=lambda: _LOG_DIR / "falhas_cnpj.csv")
    aba_excel       : int   = 0
    base_sleep      : float = 1.0
    tentativas      : int   = 3
    timeout_http    : int   = 15
    sleep_fixo      : float = 0.6
    max_concurrent  : int   = 5
    cache_ttl_dias  : int   = 30
    # Opções: "lista_nomes" | "principal" | "quantidade" | "detalhado"
    formato_socios  : Literal["lista_nomes", "principal", "quantidade", "detalhado"] = "lista_nomes"

    def __post_init__(self) -> None:
        for pasta in (_INPUT_DIR, _OUTPUT_DIR, _LOG_DIR):
            pasta.mkdir(parents=True, exist_ok=True)

CFG = Config()

# =======================
# ESTRUTURA DO CACHE
# =======================
class DadosCNPJ(NamedTuple):
    """Estrutura imutável dos dados retornados pela BrasilAPI para um CNPJ."""
    data_abertura : Optional[str]
    ano_abertura  : Optional[int]
    idade_anos    : Optional[int]
    situacao_desc : Optional[str]
    situacao_cod  : Optional[int]
    data_situacao : Optional[str]
    socios        : Optional[object]
    bairro        : Optional[str]
    setor         : Optional[str]
    porte         : Optional[str]
    ativa         : Optional[bool]

CACHE_VAZIO    = DadosCNPJ(*[None] * len(DadosCNPJ._fields))
CACHE_N_CAMPOS = len(DadosCNPJ._fields)


def cache_tem_resultado(valor) -> bool:
    """
    Determina se uma entrada de cache contém dados úteis.
    
    Retorna False para: None, CACHE_VAZIO, ou entradas com todos os campos None.
    Retorna True apenas se há pelo menos um campo preenchido (sucesso na API).
    
    Args:
        valor: Entrada do cache (tipicamente DadosCNPJ ou None)
        
    Returns:
        True se tem resultado válido, False se é falha memoizada
    """
    return isinstance(valor, DadosCNPJ) and any(campo is not None for campo in valor)

MAPA_PORTE = {
    "MICRO EMPRESA"            : "ME",
    "EMPRESA DE PEQUENO PORTE" : "EPP",
    "DEMAIS"                   : "DEMAIS",
}

warnings.filterwarnings(
    "ignore",
    message="Workbook contains no default style, apply openpyxl's default",
    category=UserWarning,
    module=r"openpyxl\.styles\.stylesheet",
)

# =======================
# UTILIDADES
# =======================
def cnpj_tem_digitos_validos(cnpj: str) -> bool:
    """
    Valida os dígitos verificadores de um CNPJ com 14 dígitos.
    
    Implementa o algoritmo de validação oficial: calcula dois dígitos verificadores
    usando pesos específicos e verifica se correspondem aos dígitos 13 e 14.
    
    Rejeita: CNPJs com formato inválido, sequências repetidas (11111111111111).
    
    Args:
        cnpj: String com 14 dígitos (ex: "12345678901234")
        
    Returns:
        True se dígitos verificadores são válidos, False caso contrário
    """
    if not re.fullmatch(r"\d{14}", cnpj):
        return False
    if cnpj == cnpj[0] * 14:
        return False

    def calcula_digito(base: str, pesos: list[int]) -> str:
        soma = sum(int(d) * p for d, p in zip(base, pesos))
        resto = soma % 11
        return "0" if resto < 2 else str(11 - resto)

    digito_1 = calcula_digito(cnpj[:12], [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2])
    digito_2 = calcula_digito(cnpj[:13], [6, 5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2])
    return cnpj[-2:] == digito_1 + digito_2


def limpa_cnpj(valor) -> Optional[str]:
    """
    Limpa e valida um CNPJ: remove caracteres, completa com zeros, valida dígitos.
    
    Aceita: strings, inteiros, floats. Remove pontos, hífens, espaços.
    Recompõe com zeros à esquerda até 14 dígitos.
    
    Args:
        valor: CNPJ em qualquer formato (pd.NA, float, string, int)
        
    Returns:
        CNPJ limpo com 14 dígitos válidos, ou None se inválido/vazio
        
    Examples:
        >>> limpa_cnpj("12.345.678/0001-90")
        '12345678000190'
        >>> limpa_cnpj(12345678000190)
        '12345678000190'
        >>> limpa_cnpj("999.999.999/9999-99")
        None  # Dígitos inválidos
    """
    if pd.isna(valor):
        return None

    texto = str(int(valor)) if isinstance(valor, float) and valor.is_integer() else str(valor)
    digitos = re.sub(r"\D", "", texto)
    if not digitos or len(digitos) > 14:
        return None

    cnpj = digitos.zfill(14)
    return cnpj if cnpj_tem_digitos_validos(cnpj) else None

def formata_data(data_str: str) -> Optional[str]:
    """
    Converte data de qualquer formato para DD/MM/YYYY.
    
    Args:
        data_str: Data em qualquer formato que pandas reconheça (ISO, BR, etc)
        
    Returns:
        Data formatada como DD/MM/YYYY, ou None se inválida
        
    Examples:
        >>> formata_data("2020-05-15")
        '15/05/2020'
    """
    try:
        dt = pd.to_datetime(data_str, errors="coerce", utc=False)
        return None if pd.isna(dt) else dt.strftime("%d/%m/%Y")
    except Exception:
        return None

def processar_socios(
    qsa_list,
    formato: Literal["lista_nomes", "principal", "quantidade", "detalhado"] = "lista_nomes",
) -> Optional[object]:
    """
    Processa a lista de sócios (QSA) retornada pela BrasilAPI.
    
    Args:
        qsa_list: Lista de dicts com 'nome_socio', 'qualificacao_socio', etc
        formato: Como retornar os sócios:
            - "quantidade": int com total de sócios
            - "principal": str com nome do primeiro sócio
            - "lista_nomes": str com todos os nomes separados por "; "
            - "detalhado": str com "Nome (Qualificação); ..." para cada sócio
        
    Returns:
        Dados dos sócios no formato solicitado, ou None se não há sócios
        
    Examples:
        >>> processar_socios([{"nome_socio": "João Silva"}], "principal")
        'João Silva'
        >>> processar_socios([{"nome_socio": "João"}, {"nome_socio": "Maria"}], "quantidade")
        2
    """
    if not isinstance(qsa_list, list):
        return None
    validos = [s for s in qsa_list if isinstance(s, dict) and s.get("nome_socio")]
    if not validos:
        return None

    if formato == "quantidade":
        return len(validos)
    if formato == "principal":
        return validos[0].get("nome_socio", "").strip() or None
    if formato == "lista_nomes":
        nomes = [s["nome_socio"].strip() for s in validos if s.get("nome_socio")]
        return "; ".join(nomes) or None
    if formato == "detalhado":
        partes = []
        for s in validos:
            nome = s.get("nome_socio", "").strip()
            qual = s.get("qualificacao_socio", "").strip()
            if nome:
                partes.append(f"{nome} ({qual})" if qual else nome)
        return "; ".join(partes) or None

    return None

async def backoff_sleep(base: float, tentativa: int) -> None:
    """
    Pausa assíncrona com backoff exponencial + jitter.
    
    Implementa: sleep(base * 2^tentativa + random(0..0.4))
    
    Exemplo com base=1.0:
        - tentativa 0: ~1.0s
        - tentativa 1: ~2.0s
        - tentativa 2: ~4.0s
        
    Args:
        base: Tempo base em segundos
        tentativa: Índice da tentativa (0, 1, 2, ...)
    """
    await asyncio.sleep(base * (2 ** tentativa) + random.uniform(0, 0.4))

# =======================
# CONSULTA API (ASYNC)
# =======================
async def consulta_cnpj(cnpj: str, sess: aiohttp.ClientSession, cfg: Config) -> DadosCNPJ:
    """
    Consulta BrasilAPI para obter dados de um CNPJ (assincronamente).
    
    Implementa retry automático: até cfg.tentativas vezes com backoff exponencial.
    Retorna CACHE_VAZIO (todos os campos None) se falhar em todas as tentativas.
    Isso memoiza a falha para evitar re-consultas infinitas.
    
    Args:
        cnpj: CNPJ com 14 dígitos (ex: "12345678901234")
        sess: aiohttp.ClientSession ativa
        cfg: Objeto Config com timeout_http, tentativas, base_sleep, etc
        
    Returns:
        DadosCNPJ com dados ou CACHE_VAZIO se falhou
        
    HTTP Status tratados:
        200: sucesso, processa resposta
        429, 500, 502, 503, 504: retry com backoff
        Outro: falha silenciosa, continua para próxima tentativa
    """
    url = f"https://brasilapi.com.br/api/cnpj/v1/{cnpj}"
    for i in range(cfg.tentativas):
        try:
            async with sess.get(url, timeout=aiohttp.ClientTimeout(total=cfg.timeout_http)) as r:
                if r.status == 200:
                    data = await r.json() or {}

                    abertura_raw = data.get("data_inicio_atividade")
                    if abertura_raw:
                        abertura_fmt = formata_data(str(abertura_raw))
                        try:
                            ano   = int(str(abertura_raw)[:4])
                            idade = datetime.now().year - ano
                        except (ValueError, TypeError):
                            ano, idade = None, None
                    else:
                        abertura_fmt = ano = idade = None

                    situacao_desc = data.get("descricao_situacao_cadastral")
                    situacao_cod  = data.get("situacao_cadastral")
                    situacao_data = data.get("data_situacao_cadastral")

                    return DadosCNPJ(
                        data_abertura = abertura_fmt,
                        ano_abertura  = ano,
                        idade_anos    = idade,
                        situacao_desc = situacao_desc,
                        situacao_cod  = situacao_cod,
                        data_situacao = formata_data(str(situacao_data)) if situacao_data else None,
                        socios        = processar_socios(data.get("qsa", []), cfg.formato_socios),
                        bairro        = data.get("bairro") or None,
                        setor         = data.get("cnae_fiscal_descricao") or None,
                        porte         = data.get("porte") or None,
                        ativa         = str(situacao_desc or "").upper() == "ATIVA",
                    )

                if r.status in (429, 500, 502, 503, 504):
                    await backoff_sleep(cfg.base_sleep, i)
                    continue

        except (aiohttp.ClientError, asyncio.TimeoutError):
            await backoff_sleep(cfg.base_sleep, i)

    return CACHE_VAZIO

# =======================
# CACHE
# =======================
def carregar_cache(caminho: Path, cfg: Config) -> dict:
    """
    Carrega cache JSON com validação de TTL.
    
    Formatos suportados:
    - Novo: {cnpj: {"dados": [...], "ts": 1716000000.0}} com validação TTL
    - Antigo: {cnpj: [...]} migra automaticamente com ts=None (reprocessa)
    
    Validações:
    - Arquivo não existe: retorna dict vazio
    - JSON inválido: loga warning, retorna dict vazio
    - Entrada expirada (ts > cache_ttl_dias): marca como None para reprocessamento
    - Formato antigo: migra automaticamente
    
    Args:
        caminho: Path para cache_cnpj.json
        cfg: Config com cache_ttl_dias
        
    Returns:
        Dict {cnpj_str: DadosCNPJ} com dados válidos
    """
    if not caminho.exists():
        return {}
    try:
        with caminho.open("r", encoding="utf-8") as f:
            raw = json.load(f)
        if not isinstance(raw, dict):
            raise ValueError("cache JSON nao contem um objeto/dicionario")

        out       : dict = {}
        expirados : int  = 0
        migrados  : int  = 0
        ttl_segundos = cfg.cache_ttl_dias * 86400
        agora = time.time()

        for k, v in raw.items():
            if isinstance(v, dict) and "dados" in v:
                ts = v.get("ts", 0)
                if agora - ts > ttl_segundos:
                    out[k] = None
                    expirados += 1
                    continue

                dados_list = v.get("dados", [])
                if isinstance(dados_list, list) and len(dados_list) >= CACHE_N_CAMPOS:
                    dados = DadosCNPJ(*dados_list[:CACHE_N_CAMPOS])
                    out[k] = dados
                else:
                    out[k] = None
                    migrados += 1

            elif isinstance(v, list) and len(v) >= CACHE_N_CAMPOS:
                dados = DadosCNPJ(*v[:CACHE_N_CAMPOS])
                out[k] = dados
                migrados += 1

            else:
                out[k] = None
                migrados += 1

        if expirados:
            logger.warning(f"Cache: {expirados} entrada(s) expirada(s) (TTL={cfg.cache_ttl_dias} dias)")
        if migrados:
            logger.warning(f"Cache: {migrados} entrada(s) em formato antigo, serao reprocessadas")

        return out
    except Exception as e:
        logger.warning(f"Falha ao carregar cache ({e}). Iniciando cache vazio.")
        return {}

def salvar_cache(caminho: Path, cache: dict, max_tentativas: int = 3) -> None:
    """
    Salva cache em JSON com timestamps e retry automático.
    
    Novo formato: {cnpj: {"dados": [...], "ts": 1716000000.0}}
    
    Implementa 3 tentativas com exponential backoff:
        0: imediata
        1: 0.5s
        2: 1.0s
    
    Distingue entre erros retryable (IOError, OSError) e críticos.
    Levanta ValueError na última falha para alertar o programa.
    
    Args:
        caminho: Path para cache_cnpj.json
        cache: Dict {cnpj_str: DadosCNPJ} a salvar
        max_tentativas: Quantas vezes tentar antes de desistir
        
    Raises:
        ValueError: Se não conseguir salvar após todas as tentativas
    """
    dados = {
        k: {"dados": list(v), "ts": time.time()}
        for k, v in cache.items() if isinstance(v, DadosCNPJ)
    }

    for tentativa in range(max_tentativas):
        try:
            with caminho.open("w", encoding="utf-8") as f:
                json.dump(dados, f, ensure_ascii=False)
            return
        except (IOError, OSError) as e:
            if tentativa < max_tentativas - 1:
                tempo_espera = 0.5 * (2 ** tentativa)
                logger.warning(f"Falha ao salvar cache (tentativa {tentativa + 1}/{max_tentativas}), aguardando {tempo_espera:.1f}s...")
                time.sleep(tempo_espera)
            else:
                logger.error(f"Falha ao salvar cache apos {max_tentativas} tentativas: {e}")
                raise ValueError(f"Cache nao pode ser salvo: {e}") from e
        except Exception as e:
            logger.error(f"Erro inesperado ao salvar cache: {e}")
            raise ValueError(f"Cache nao pode ser salvo: {e}") from e

# =======================
# EXCEL
# =======================
def salvar_excel_com_fallback(df: pd.DataFrame, caminho: Path) -> None:
    """
    Salva DataFrame em Excel com fallback automático.
    
    Tenta salvar com xlsxwriter (melhor para formatação).
    Se falhar, usa engine padrão do pandas.
    
    Args:
        df: DataFrame a salvar
        caminho: Path para arquivo .xlsx
    """
    try:
        df.to_excel(caminho, index=False, engine="xlsxwriter")
    except Exception:
        df.to_excel(caminho, index=False)

# =======================
# VALIDAÇÕES
# =======================
def verificar_colunas_obrigatorias(df: pd.DataFrame, colunas: list) -> None:
    """
    Verifica se DataFrame possui colunas obrigatórias.
    
    Args:
        df: DataFrame a validar
        colunas: Lista de nomes de colunas obrigatórias
        
    Raises:
        ValueError: Se alguma coluna obrigatória estiver faltando
    """
    faltantes = [c for c in colunas if c not in df.columns]
    if faltantes:
        raise ValueError(
            f"Colunas obrigatorias nao encontradas: {', '.join(faltantes)}\n"
            f"Colunas disponiveis: {', '.join(df.columns)}"
        )

def validar_estrutura_entrada(df: pd.DataFrame) -> None:
    """Loga estatísticas do arquivo de entrada."""
    logger.info("VALIDACAO DO ARQUIVO DE ENTRADA")
    logger.info(f"  Total de linhas: {len(df)}")
    logger.info(f"  CNPJs preenchidos: {df['cnpj'].notna().sum()}")
    logger.info(f"  CNPJs vazios: {df['cnpj'].isna().sum()}")
    logger.info(f"  Colunas encontradas: {len(df.columns)}")

def limpar_e_validar_cnpjs(df: pd.DataFrame) -> list:
    """
    Limpa e valida CNPJs, retorna lista de CNPJs únicos válidos.
    
    Cria coluna 'cnpj_limpo' com CNPJs validados.
    Loga: original, preenchidos, válidos, descartados, duplicados, únicos.
    
    Args:
        df: DataFrame com coluna 'cnpj'
        
    Returns:
        Lista de CNPJs únicos válidos (pronto para consultar)
    """
    df["cnpj_limpo"] = df["cnpj"].apply(limpa_cnpj)
    preenchidos = df["cnpj"].dropna().astype(str).str.strip()
    validos     = df["cnpj_limpo"].dropna().astype(str)
    unicos      = validos.unique().tolist()

    logger.info("LIMPEZA E VALIDACAO DE CNPJs")
    logger.info(f"  CNPJs originais: {len(df)}")
    logger.info(f"  CNPJs preenchidos: {df['cnpj'].notna().sum()}")
    logger.info(f"  CNPJs validos apos limpeza: {len(validos)}")
    logger.info(f"  CNPJs descartados (invalidos): {len(preenchidos) - len(validos)}")
    logger.info(f"  CNPJs duplicados removidos: {len(validos) - len(unicos)}")
    logger.info(f"  CNPJs validos unicos: {len(unicos)}")
    return unicos

def verificar_cache_existente(cache: dict, cnpjs_validos: list) -> list:
    """
    Separa CNPJs em dois grupos: em cache vs a consultar.
    
    Args:
        cache: Dict {cnpj: DadosCNPJ}
        cnpjs_validos: Lista de CNPJs para processar
        
    Returns:
        Lista de CNPJs que ainda precisam ser consultados
    """
    em_cache    = [c for c in cnpjs_validos if cache_tem_resultado(cache.get(c))]
    a_consultar = [c for c in cnpjs_validos if not cache_tem_resultado(cache.get(c))]
    pct = len(em_cache) / len(cnpjs_validos) * 100 if cnpjs_validos else 0.0
    logger.info("STATUS DO CACHE")
    logger.info(f"  CNPJs em cache valido: {len(em_cache)} ({pct:.1f}%)")
    logger.info(f"  CNPJs a consultar: {len(a_consultar)}")
    return a_consultar

# =======================
# APLICAR RESULTADOS
# =======================
def aplicar_resultados_ao_dataframe(df: pd.DataFrame, cache: dict) -> pd.DataFrame:
    """
    Aplica dados de cache ao DataFrame original.
    
    Para cada CNPJ_limpo, busca dados no cache e cria colunas:
    DataAbertura, AnoFundacao, IdadeEmpresa, SituacaoDesc, SituacaoCod,
    DataSituacao, Socios, Bairro, Setor, Porte, Ativa
    
    Converte tipos (int, bool) e aplica mapeamento de porte (ME, EPP, DEMAIS).
    Força CNPJ como texto para evitar conversão automática do Excel.
    
    Args:
        df: DataFrame com coluna 'cnpj_limpo'
        cache: Dict {cnpj: DadosCNPJ}
        
    Returns:
        DataFrame modificado com novas colunas
    """
    def resolve(cnpj) -> Optional[DadosCNPJ]:
        if pd.isna(cnpj):
            return None
        r = cache.get(cnpj)
        return r if isinstance(r, DadosCNPJ) else None

    dados = df["cnpj_limpo"].apply(resolve)

    df["DataAbertura"] = dados.apply(lambda d: d.data_abertura if d else None)
    df["AnoFundacao"]  = dados.apply(lambda d: d.ano_abertura  if d else None)
    df["IdadeEmpresa"] = dados.apply(lambda d: d.idade_anos    if d else None)
    df["SituacaoDesc"] = dados.apply(lambda d: d.situacao_desc if d else None)
    df["SituacaoCod"]  = dados.apply(lambda d: d.situacao_cod  if d else None)
    df["DataSituacao"] = dados.apply(lambda d: d.data_situacao if d else None)
    df["Socios"]       = dados.apply(lambda d: d.socios        if d else None)
    df["Bairro"]       = dados.apply(lambda d: d.bairro        if d else None)
    df["Setor"]        = dados.apply(lambda d: d.setor         if d else None)
    df["Porte"]        = dados.apply(lambda d: d.porte         if d else None)
    df["Ativa"]        = dados.apply(lambda d: d.ativa         if d else None)

    df["Porte"] = df["Porte"].apply(
        lambda v: MAPA_PORTE.get(str(v).upper().strip(), v) if pd.notna(v) else v
    )
    for col in ["AnoFundacao", "IdadeEmpresa", "SituacaoCod"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    df["cnpj"] = df["cnpj"].astype(str)
    df["cnpj_limpo"] = df["cnpj_limpo"].astype(str)

    return df

# =======================
# DASHBOARD HTML
# =======================
def gerar_dashboard(df: pd.DataFrame, caminho: Path, tempo_total: float) -> None:
    """
    Gera relatório HTML visual com estatísticas e gráficos.
    
    Seções:
    - Cards com métricas principais (total, únicos, % ativas, com sócios)
    - Gráfico pizza: Situação Cadastral
    - Gráfico barras: Distribuição de Porte
    - Tabela: Top-10 empresas mais recentes
    - Footer: data/hora e tempo de processamento
    
    Args:
        df: DataFrame com dados processados
        caminho: Path para salvar dashboard.html
        tempo_total: Tempo total de processamento em segundos
    """
    if len(df) == 0:
        logger.warning("DataFrame vazio, dashboard nao sera gerado")
        return

    total = len(df)
    qtd_ativas = int(df["Ativa"].eq(True).sum())
    pct_ativas = qtd_ativas / total * 100 if total > 0 else 0.0
    qtd_com_socios = int(df["Socios"].notna().sum())

    situacao_counts = df["SituacaoDesc"].value_counts()
    situacao_labels = list(situacao_counts.index)
    situacao_values = [int(v) for v in situacao_counts.values]

    porte_counts = df["Porte"].value_counts()
    porte_labels = list(porte_counts.index)
    porte_values = [int(v) for v in porte_counts.values]

    if "dataAtendimento" in df.columns:
        df_temp = df.copy()
        df_temp["_dt_parse"] = pd.to_datetime(df_temp["dataAtendimento"], errors="coerce")
        top10 = df_temp.nlargest(10, "_dt_parse")[
            ["dataAtendimento", "razaoSocial", "cnpj", "Bairro", "SituacaoDesc"]
        ].fillna("")
    else:
        top10 = df.head(10)[
            ["razaoSocial", "cnpj", "Bairro", "SituacaoDesc"]
        ].fillna("")

    table_rows = []
    for _, row in top10.iterrows():
        if "dataAtendimento" in top10.columns:
            data = str(row.get("dataAtendimento", ""))
        else:
            data = ""
        razao = str(row.get("razaoSocial", ""))
        cnpj_val = str(row.get("cnpj", ""))
        bairro = str(row.get("Bairro", ""))
        situacao = str(row.get("SituacaoDesc", ""))
        table_rows.append(f"<tr><td>{data}</td><td>{razao}</td><td>{cnpj_val}</td><td>{bairro}</td><td>{situacao}</td></tr>")

    table_html = "\n                ".join(table_rows)

    html = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>CNPJ Dashboard</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js@3"></script>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; background: #f5f5f5; padding: 20px; }}
        .container {{ max-width: 1200px; margin: 0 auto; background: white; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); padding: 30px; }}
        h1 {{ color: #333; margin-bottom: 10px; }}
        .subtitle {{ color: #666; font-size: 14px; margin-bottom: 30px; }}
        .cards {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin-bottom: 40px; }}
        .card {{ background: linear-gradient(135deg, #007bff 0%, #0056b3 100%); color: white; padding: 20px; border-radius: 8px; text-align: center; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }}
        .card:nth-child(2) {{ background: linear-gradient(135deg, #28a745 0%, #1e7e34 100%); }}
        .card:nth-child(3) {{ background: linear-gradient(135deg, #ffc107 0%, #cc9a00 100%); }}
        .card:nth-child(4) {{ background: linear-gradient(135deg, #17a2b8 0%, #117a8b 100%); }}
        .card h3 {{ font-size: 13px; opacity: 0.95; margin-bottom: 10px; font-weight: 600; }}
        .card .value {{ font-size: 36px; font-weight: bold; margin: 0; }}
        .sections {{ display: grid; grid-template-columns: 1fr 1fr; gap: 30px; margin-bottom: 40px; }}
        .section-title {{ font-size: 18px; font-weight: bold; color: #333; margin-bottom: 15px; }}
        table {{ width: 100%; border-collapse: collapse; }}
        th, td {{ padding: 12px; text-align: left; border-bottom: 1px solid #ddd; }}
        th {{ background: #f9f9f9; font-weight: 600; color: #333; }}
        tr:hover {{ background: #f5f5f5; }}
        .footer {{ text-align: center; margin-top: 40px; padding-top: 20px; border-top: 1px solid #ddd; color: #999; font-size: 13px; }}
        @media (max-width: 768px) {{
            .sections {{ grid-template-columns: 1fr; }}
            .cards {{ grid-template-columns: repeat(2, 1fr); }}
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>CNPJ Consultation Dashboard</h1>
        <div class="subtitle">Gerado em {datetime.now().strftime('%d/%m/%Y %H:%M:%S')} | Tempo de processamento: {tempo_total:.1f}s</div>
        
        <div class="cards">
            <div class="card"><h3>Total de Registros</h3><div class="value">{total}</div></div>
            <div class="card"><h3>CNPJs Unicos</h3><div class="value">{df['cnpj_limpo'].nunique()}</div></div>
            <div class="card"><h3>% Ativas</h3><div class="value">{pct_ativas:.1f}%</div></div>
            <div class="card"><h3>Com Socios</h3><div class="value">{qtd_com_socios}</div></div>
        </div>
        
        <div class="sections">
            <div>
                <div class="section-title">Situacao Cadastral</div>
                <canvas id="situacaoChart"></canvas>
            </div>
            <div>
                <div class="section-title">Distribuicao de Porte</div>
                <canvas id="porteChart"></canvas>
            </div>
        </div>
        
        <div>
            <div class="section-title">Top 10 Empresas Mais Recentes</div>
            <table>
                <tr><th>Data</th><th>Razao Social</th><th>CNPJ</th><th>Bairro</th><th>Situacao</th></tr>
                {table_html}
            </table>
        </div>
        
        <div class="footer">Relatorio gerado automaticamente pelo CNPJ Consultation System</div>
    </div>
    
    <script>
        const situacaoCtx = document.getElementById('situacaoChart').getContext('2d');
        new Chart(situacaoCtx, {{
            type: 'doughnut',
            data: {{
                labels: {json.dumps(situacao_labels, ensure_ascii=False)},
                datasets: [{{
                    data: {json.dumps(situacao_values)},
                    backgroundColor: ['#28a745', '#dc3545', '#ffc107', '#17a2b8', '#6c757d'],
                    borderColor: '#fff',
                    borderWidth: 2
                }}]
            }},
            options: {{
                responsive: true,
                maintainAspectRatio: true,
                plugins: {{
                    legend: {{ position: 'bottom', labels: {{ padding: 15, font: {{ size: 12 }} }} }}
                }}
            }}
        }});
        
        const porteCtx = document.getElementById('porteChart').getContext('2d');
        new Chart(porteCtx, {{
            type: 'bar',
            data: {{
                labels: {json.dumps(porte_labels, ensure_ascii=False)},
                datasets: [{{
                    label: 'Quantidade',
                    data: {json.dumps(porte_values)},
                    backgroundColor: '#007bff',
                    borderColor: '#0056b3',
                    borderWidth: 1
                }}]
            }},
            options: {{
                responsive: true,
                maintainAspectRatio: true,
                indexAxis: 'y',
                plugins: {{
                    legend: {{ display: true, labels: {{ font: {{ size: 12 }} }} }}
                }},
                scales: {{
                    x: {{ beginAtZero: true, ticks: {{ stepSize: 1 }} }}
                }}
            }}
        }});
    </script>
</body>
</html>
    """

    try:
        with open(caminho, 'w', encoding='utf-8') as f:
            f.write(html)
        logger.info(f"Dashboard salvo: {caminho}")
    except Exception as e:
        logger.error(f"Falha ao salvar dashboard: {e}")

# =======================
# RELATÓRIO FINAL
# =======================
def gerar_relatorio_final(df: pd.DataFrame, cnpjs_validos: list, falhas: list) -> None:
    """
    Loga estatísticas finais: distribuição por status, porte, falhas.
    """
    total      = len(df)
    qtd_ativas = df["Ativa"].eq(True).sum()

    logger.info("RELATORIO FINAL DE PROCESSAMENTO")
    logger.info(f"  Total de registros: {total}")
    logger.info(f"  CNPJs unicos: {len(cnpjs_validos)}")
    logger.info(f"  Com data de abertura: {df['DataAbertura'].notna().sum()}")
    logger.info(f"  Com situacao cadastral: {df['SituacaoDesc'].notna().sum()}")
    if total:
        logger.info(f"  Empresas ATIVAS: {qtd_ativas} ({qtd_ativas / total * 100:.1f}%)")
    logger.info(f"  Com socios: {df['Socios'].notna().sum()}")
    logger.info(f"  Com bairro: {df['Bairro'].notna().sum()}")
    logger.info(f"  Com setor (CNAE): {df['Setor'].notna().sum()}")
    logger.info(f"  Com porte: {df['Porte'].notna().sum()}")

    logger.info("DISTRIBUICAO - Situacao Cadastral")
    for sit, cnt in df["SituacaoDesc"].value_counts().items():
        logger.info(f"  {sit}: {cnt} ({cnt / total * 100:.1f}%)")

    logger.info("DISTRIBUICAO - Porte")
    for p, cnt in df["Porte"].value_counts().items():
        logger.info(f"  {p}: {cnt} ({cnt / total * 100:.1f}%)")

    n_falhas      = len(falhas)
    n_total_cnpjs = len(cnpjs_validos)
    if falhas:
        pct = n_falhas / n_total_cnpjs * 100 if n_total_cnpjs else 0.0
        logger.warning(f"CNPJs sem retorno: {n_falhas} ({pct:.1f}%)")
    else:
        logger.info("Sem falhas detectadas!")

def gerar_excel_resumido(df: pd.DataFrame, caminho: Path) -> None:
    """
    Gera arquivo Excel resumido com colunas selecionadas.
    
    Colunas: dataAtendimento, razaoSocial, nomeFantasia, cnpj, Bairro, Porte,
    AnoFundacao, IdadeEmpresa, SituacaoDesc, Socios
    
    Args:
        df: DataFrame com dados completos
        caminho: Path para salvar arquivo resumido
    """
    COLUNAS = [
        "dataAtendimento", "razaoSocial", "nomeFantasia", "cnpj",
        "Bairro", "Porte", "AnoFundacao", "IdadeEmpresa", "SituacaoDesc", "Socios",
    ]
    existentes = [c for c in COLUNAS if c in df.columns]
    faltantes  = [c for c in COLUNAS if c not in df.columns]

    if faltantes:
        logger.warning(f"Colunas ausentes (serao ignoradas): {', '.join(faltantes)}")

    df_resumido = df[existentes].copy()
    df_resumido["cnpj"] = df_resumido["cnpj"].astype(str)
    n           = len(df_resumido)

    logger.info("Arquivo resumido - preenchimento por coluna")
    for col in existentes:
        preench = df_resumido[col].notna().sum()
        pct     = preench / n * 100 if n else 0.0
        logger.info(f"  {col:<20}: {preench}/{n} ({pct:.1f}%)")

    try:
        df_resumido.to_excel(caminho, index=False, engine="openpyxl")
        logger.info(f"Arquivo resumido salvo: {caminho}")
    except Exception as e:
        fallback = caminho.with_suffix(".csv")
        df_resumido.to_csv(fallback, index=False, encoding="utf-8-sig")
        logger.warning(f"Falha ao salvar Excel: {e}")
        logger.info(f"Salvo como CSV: {fallback}")

# =======================
# WORKER ASYNC
# =======================
async def worker_consulta(
    cnpj: str,
    sess: aiohttp.ClientSession,
    cfg: Config,
    semaphore: asyncio.Semaphore,
    cache: dict,
    results: list,
    lock: asyncio.Lock,
) -> None:
    """
    Worker que consulta um CNPJ assincronamente com controle de concorrência.
    
    - Aguarda slot no semáforo (max_concurrent)
    - Consulta CNPJ via API
    - Armazena resultado no cache (thread-safe com lock)
    - Aguarda sleep_fixo antes de liberar slot
    
    Args:
        cnpj: CNPJ a consultar
        sess: ClientSession ativa
        cfg: Config com parâmetros
        semaphore: asyncio.Semaphore para limitar concorrência
        cache: Dict mutável {cnpj: DadosCNPJ} a atualizar
        results: Lista mutável para rastrear CNPJs processados
        lock: asyncio.Lock para acesso thread-safe ao cache
    """
    async with semaphore:
        try:
            dados = await consulta_cnpj(cnpj, sess, cfg)
            async with lock:
                cache[cnpj] = dados
                results.append(cnpj)
        except Exception as e:
            logger.debug(f"Erro ao consultar {cnpj}: {e}")
            async with lock:
                cache[cnpj] = CACHE_VAZIO
                results.append(cnpj)

        await asyncio.sleep(cfg.sleep_fixo)

# =======================
# PIPELINE PRINCIPAL (ASYNC)
# =======================
async def main() -> None:
    """
    Pipeline principal de consulta de CNPJs.
    
    Fluxo:
    1. Carrega arquivo de entrada
    2. Valida e limpa CNPJs
    3. Carrega cache anterior (com validação TTL)
    4. Consulta CNPJs faltantes (assincronamente, até max_concurrent paralelos)
    5. Aplica resultados ao DataFrame
    6. Salva arquivo de saída completo
    7. Gera arquivo resumido
    8. Gera dashboard HTML
    9. Loga relatório final
    
    Erros são logados mas não interrompem: falhas de API são memoizadas,
    falhas de Excel usam fallback CSV.
    """
    tempo_inicio = time.time()

    try:
        logger.info(f"Lendo arquivo: {CFG.arquivo_entrada}")
        df = pd.read_excel(CFG.arquivo_entrada, sheet_name=CFG.aba_excel)

        verificar_colunas_obrigatorias(df, ["cnpj"])
        validar_estrutura_entrada(df)

        cnpjs_validos = limpar_e_validar_cnpjs(df)
        if not cnpjs_validos:
            raise ValueError("Nenhum CNPJ valido encontrado apos limpeza e validacao!")

        cache           = carregar_cache(CFG.arquivo_cache, CFG)
        cnpjs_pendentes = verificar_cache_existente(cache, cnpjs_validos)

        if cnpjs_pendentes:
            total = len(cnpjs_pendentes)
            logger.info(f"Consultando {total} CNPJs na BrasilAPI (ate {CFG.max_concurrent} paralelos)...")
            logger.info(f"Formato de socios: {CFG.formato_socios}")

            connector = aiohttp.TCPConnector(limit_per_host=CFG.max_concurrent)
            timeout = aiohttp.ClientTimeout(total=CFG.timeout_http * CFG.tentativas * 3)

            async with aiohttp.ClientSession(connector=connector, timeout=timeout) as sess:
                sess.headers.update({
                    "User-Agent": "Mozilla/5.0 (compatible; CNPJBatchBot/1.0; +https://brasilapi.com.br)"
                })

                semaphore = asyncio.Semaphore(CFG.max_concurrent)
                lock = asyncio.Lock()
                results = []

                tasks = [
                    worker_consulta(cnpj, sess, CFG, semaphore, cache, results, lock)
                    for cnpj in cnpjs_pendentes
                ]

                for i, task in enumerate(asyncio.as_completed(tasks), start=1):
                    await task
                    if i % 25 == 0 or i == total:
                        salvar_cache(CFG.arquivo_cache, cache)
                        logger.info(f"Progresso: {i}/{total} CNPJs processados, cache salvo")

                logger.info(f"Concluido: {len(results)}/{total} CNPJs consultados")
        else:
            logger.info("Todos os CNPJs ja estao em cache valido!")

        logger.info("Aplicando resultados ao DataFrame...")
        df = aplicar_resultados_ao_dataframe(df, cache)

        if "dataAtendimento" in df.columns:
            logger.info("Ordenando por data de atendimento...")
            df["_dt_sort"] = pd.to_datetime(df["dataAtendimento"], errors="coerce")
            df = df.sort_values("_dt_sort", ascending=False, na_position="last")
            df = df.drop("_dt_sort", axis=1).reset_index(drop=True)

        falhas = [c for c in cnpjs_validos if not cache_tem_resultado(cache.get(c))]

        if falhas:
            pd.DataFrame({"cnpj_limpo": falhas}).to_csv(
                CFG.arquivo_falhas, index=False, encoding="utf-8-sig"
            )
            logger.warning(f"{len(falhas)} CNPJs sem retorno. Log: {CFG.arquivo_falhas}")

        salvar_cache(CFG.arquivo_cache, cache)
        salvar_excel_com_fallback(df, CFG.arquivo_saida)
        logger.info(f"Arquivo completo salvo: {CFG.arquivo_saida}")

        gerar_relatorio_final(df, cnpjs_validos, falhas)

        logger.info("GERANDO ARQUIVO RESUMIDO")
        arquivo_resumido = CFG.arquivo_saida.with_name(
            CFG.arquivo_saida.stem + "_RESUMIDO" + CFG.arquivo_saida.suffix
        )
        gerar_excel_resumido(df, arquivo_resumido)

        tempo_total = time.time() - tempo_inicio
        logger.info(f"GERANDO DASHBOARD HTML")
        arquivo_dashboard = CFG.arquivo_saida.with_name("dashboard.html")
        gerar_dashboard(df, arquivo_dashboard, tempo_total)

        logger.info("PROCESSAMENTO CONCLUIDO COM SUCESSO!")
        logger.info(f"Arquivos gerados:")
        logger.info(f"  1. Completo: {CFG.arquivo_saida}")
        logger.info(f"  2. Resumido: {arquivo_resumido}")
        logger.info(f"  3. Dashboard: {arquivo_dashboard}")
        if falhas:
            logger.info(f"  4. Falhas: {CFG.arquivo_falhas}")

    except FileNotFoundError as e:
        logger.error(f"Arquivo nao encontrado: {e}")
    except ValueError as e:
        logger.error(f"Erro de validacao: {e}")
    except Exception as e:
        logger.exception(f"Erro inesperado: {type(e).__name__}: {e}")

if __name__ == "__main__":
    asyncio.run(main())